# Prepare ISIC 2018 dataset for VLM training

Convert the official **ISIC 2018 Task 3** dump (HAM10000 train + official val/test) into canonical JSONL.

Kept as an **experimental / exploration** resource. It is **not** the primary dataset of the main study (PAD-UFES-20).

- **Source:** `data/datasets/ISIC18/` — `ISIC2018_Task3_{Training,Validation,Test}_{Input,GroundTruth}/`
- **Output:** `data/processed/isic18/clinical_context/`
- **Splits:** official Task 3 partitions (not reshuffled)
- **Labels:** one-hot `MEL/NV/BCC/AKIEC/BKL/DF/VASC` → closed codes `MEL/NEV/BCC/ACK/BKL/DF/VASC`
- **Metadata:** `diagnosis_confirm_type` on train only (lesion groupings). No age/sex/site in this dump.
- **Malignancy:** derived from the Task 3 class (`AKIEC` omitted as premalignant)


## 1. Setup


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise FileNotFoundError(f"Could not find repo root (src/) from {Path.cwd()}")

sys.path.insert(0, str(ROOT / "src"))

from vlm_ft.data.canonical import isic18_image_rel, isic18_prompt_and_label
from vlm_ft.data.prepare import (
    count_existing_images,
    load_isic18,
    preview_processed,
    print_source_eda,
    rel_to_root,
    require_source,
    rows_to_samples,
    write_processed_dataset,
)

CURRENT_SOURCES = "PAD, ISIC18, HC, Derm1M, MILK10K"

ISIC_ROOT = ROOT / "data/datasets/ISIC18"
OUT_DIR = ROOT / "data/processed/isic18/clinical_context"
require_source(ISIC_ROOT, expected=CURRENT_SOURCES)


## 2. Load


In [ ]:
splits = load_isic18(ISIC_ROOT)
for name, frame in splits.items():
    print(name, frame.shape)
splits["train"].head()


## 3. EDA


In [ ]:
eda = pd.concat(splits.values(), ignore_index=True)
for name, frame in splits.items():
    ok, total = count_existing_images(frame, "image_rel", ISIC_ROOT)
    print(f"{name} images on disk: {ok}/{total}")

print_source_eda(
    eda,
    label_col="code",
    metadata_cols=["diagnosis_confirm_type", "malignancy"],
    extra={"task3_code": eda["task3_code"].value_counts()},
)


## 4. Convert to canonical JSONL


In [ ]:
ISIC_ROOT_REL = rel_to_root(ISIC_ROOT, ROOT)
samples = {
    split: rows_to_samples(
        frame,
        ISIC_ROOT,
        prompt_and_label=isic18_prompt_and_label,
        image_rel=isic18_image_rel,
    )
    for split, frame in splits.items()
}
write_processed_dataset(
    name="isic18/clinical_context",
    out_dir=OUT_DIR,
    split_samples=samples,
    image_root_rel=ISIC_ROOT_REL,
)


## 5. Validate and preview


In [ ]:
preview_processed(OUT_DIR, "isic18/clinical_context")
